# Glüten — Stratified bias audit (v3)

**Task:** run the v2 merged Gemma 4 E4B Marsh classifier on the 400-patch IBDColEpi held-out test set, and save **per-patch predictions plus metadata** so we can compute stratified metrics offline.

**Why a separate notebook:** v1 trained and reported aggregate accuracy. v2 retrained against the official Gemma 4 base, merged to fp16, deployed to Modal. **v3 is the audit** — it does no training, no merging, no deploy. Only inference + CSV save. Keeps the audit data lineage clean and reproducible from a single notebook.

**What this notebook produces:**
- `/kaggle/working/marsh_stratified_predictions.csv` — one row per test patch, columns `image_path, wsi_id, epi_frac, true_marsh, predicted_marsh`
- Aggregate `classification_report` printed for cross-check against v2 metrics (sanity)

**What the local audit script (`scripts/stratified_audit.py`) then does** with that CSV:
- accuracy + F1 per Marsh class
- accuracy per WSI source slide (tests slide-level overfitting)
- accuracy per `epi_frac` quintile (tests behaviour on edge vs mid-coverage patches)
- a 3-panel PNG figure saved to `results/marsh_stratified.png`
- a `results/marsh_stratified.csv` summary table

**Honest scope:** the IBDColEpi dataset is single-site (NTNU / St. Olavs, Trondheim) with no patient demographics. We cannot stratify by ancestry, sex, or age — those labels do not exist in the source. The audit's main finding is precisely that **the data needed to validate this model on non-European populations does not currently exist in publicly available CD biopsy datasets**. This notebook collects what evidence we can; the absent strata are the equity finding.

**Runtime:** Kaggle free T4, ~10-15 minutes (inference only, no training).

**Prereqs:**
- Kaggle secret `HF_TOKEN` (license accepted on `google/gemma-4-E4B-it`)
- Attached dataset `faithogun/gluten-ibdcolepi-sample`
- The merged v2 checkpoint at `faith-ogun/gluten-gemma4-marsh-merged` (pushed by v2 notebook)


## 1 · Setup and auth


In [ ]:
%%capture
!pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q --no-deps cut-cross-entropy
!pip install -q --upgrade transformers
!pip install -q bitsandbytes trl peft accelerate tifffile

import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
import transformers; print('Transformers:', transformers.__version__)
import PIL; print('Pillow:', PIL.__version__)


## 2 · Load the merged v2 model from HuggingFace

The merged checkpoint is ~15 GB. It loads with vanilla `AutoModelForImageTextToText.from_pretrained` — no PEFT, no bitsandbytes. We use `device_map='auto'` so accelerate spreads it across GPU + CPU if it doesn't fit GPU alone (T4 has 14.5 GB, model is ~15 GB so some CPU offload is expected — inference is slower but still completes).


In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MERGED_REPO = 'faith-ogun/gluten-gemma4-marsh-merged'

processor = AutoProcessor.from_pretrained(MERGED_REPO, token=os.environ['HF_TOKEN'])
model = AutoModelForImageTextToText.from_pretrained(
    MERGED_REPO,
    dtype=torch.float16,    # T4 doesn't support bf16, use fp16
    device_map='auto',      # spread across GPU+CPU if needed
    token=os.environ['HF_TOKEN'],
)
model.eval()
print('Merged Gemma 4 E4B + Marsh LoRA loaded. Device map:', model.hf_device_map if hasattr(model, 'hf_device_map') else 'cuda')


## 3 · Re-build the pseudo-Marsh labels (same algorithm as v1/v2)

We compute `epi_frac` per patch and quantile-bin into 4 classes. Identical to v2 so predictions are comparable.


In [ ]:
import pandas as pd, numpy as np
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/faithogun/gluten-ibdcolepi-sample')
PATCHES = DATA_ROOT / 'patch-dataset-HE-sampled'
SEED_CSV = DATA_ROOT / 'marsh_pseudo_labels-sampled.csv'

seed = pd.read_csv(SEED_CSV)

def read_mask(label_path):
    return np.array(Image.open(PATCHES / label_path))
def read_image(image_path):
    return Image.open(PATCHES / image_path).convert('RGB')

seed['epi_frac'] = [float((read_mask(r['label_path']) > 0).mean())
                    for _, r in tqdm(seed.iterrows(), total=len(seed))]

q = seed['epi_frac'].quantile([0.25, 0.5, 0.75]).values
def bin_fn(f):
    if f >= q[2]: return 'Marsh-0'
    if f >= q[1]: return 'Marsh-1'
    if f >= q[0]: return 'Marsh-3a'
    return 'Marsh-3b'
seed['marsh_bin'] = seed['epi_frac'].apply(bin_fn)

test_df = seed[seed['split'] == 'Testset'].sample(min(400, (seed['split'] == 'Testset').sum()), random_state=42).reset_index(drop=True)
print('Test patches:', len(test_df))
print('Class distribution:')
print(test_df['marsh_bin'].value_counts())


## 4 · Run inference and save per-patch predictions + metadata

Each row of the output CSV carries everything needed to compute stratified metrics offline: the image path, the WSI source slide, the epi_frac value, the true Marsh label, and the model's prediction. Estimated ~9 minutes for 400 patches at ~1.3 s each.


In [ ]:
INSTRUCTION = (
    'You are a histopathology assistant. Classify the Marsh grade of this HE-stained '
    'intestinal biopsy patch. Respond with exactly one of: Marsh-0, Marsh-1, Marsh-3a, Marsh-3b.'
)
VALID = ['Marsh-0', 'Marsh-1', 'Marsh-3a', 'Marsh-3b']

def load_patch(image_path):
    return read_image(image_path).resize((224, 224))

def parse_marsh(text):
    norm = text.strip().replace(' ', '').replace('_', '-').lower()
    for c in VALID:
        if c.lower() in norm:
            return c
    return 'unknown'

rows = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img = load_patch(row['image_path'])
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': img},
        {'type': 'text', 'text': INSTRUCTION}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
    ).to('cuda')
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False, temperature=0)
    txt = processor.decode(out[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    rows.append({
        'image_path': row['image_path'],
        'wsi_id': row['wsi_id'],
        'epi_frac': row['epi_frac'],
        'true_marsh': row['marsh_bin'],
        'predicted_marsh': parse_marsh(txt),
        'raw_output': txt,
    })

preds_df = pd.DataFrame(rows)
OUT = '/kaggle/working/marsh_stratified_predictions.csv'
preds_df.to_csv(OUT, index=False)
print(f'Saved {len(preds_df)} rows to {OUT}')
print(preds_df.head())


## 5 · Sanity: aggregate metrics should match v2

v2 reported 70% accuracy / Marsh-3b F1 = 0.84. This block runs the same `classification_report` so we can confirm the merged model produces equivalent predictions to the in-kernel adapter+base used at v2 training time.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(
    preds_df['true_marsh'], preds_df['predicted_marsh'],
    labels=VALID, zero_division=0,
))
print(confusion_matrix(preds_df['true_marsh'], preds_df['predicted_marsh'], labels=VALID))


## 6 · Download the predictions CSV

Pull `/kaggle/working/marsh_stratified_predictions.csv` from the file pane (right sidebar). Drop it into the project's `results/` folder locally as `marsh_stratified_predictions.csv`, then run `python scripts/stratified_audit.py` to generate the figures and audit summary.


In [ ]:
!ls -lh /kaggle/working/marsh_stratified_predictions.csv
